In [1]:
!nvidia-smi

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Tue Sep 15 14:45:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Clone HERMES

In [2]:
%cd /kaggle/working

!git clone https://github.com/haowei-freesky/HERMES.git

%cd /kaggle/working/HERMES

!git rev-parse HEAD

/kaggle/working
fatal: destination path 'HERMES' already exists and is not an empty directory.
/kaggle/working/HERMES
8d699b16a6bedb9086c1b39ec4253c6a1d1ce789


In [3]:
!git rev-parse HEAD

8d699b16a6bedb9086c1b39ec4253c6a1d1ce789


Install the LLaVA dependencies without replacing Kaggle's CUDA/PyTorch stack

In [4]:
import sys
import torch
import transformers
import numpy
import pandas
import accelerate
import huggingface_hub

print("Python:", sys.version)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

from transformers import (
    LlavaOnevisionForConditionalGeneration,
    AutoProcessor,
    DynamicCache,
)

print("Core LLaVA/HERMES imports: OK")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.10.0+cu128
transformers: 4.45.0.dev0
numpy: 2.3.3
pandas: 2.3.2
accelerate: 1.11.0
huggingface_hub: 0.34.4
Core LLaVA/HERMES imports: OK


In [6]:
!python -m pip install -q --no-deps \
    decord==0.6.0 \
    logzero==1.7.0

In [7]:
import decord
import logzero

print("decord:", decord.__version__)
print("logzero import: OK")

decord: 0.6.0
logzero import: OK


Test importing HERMES

In [8]:
import sys

sys.path.insert(0, "/kaggle/working/HERMES")

from inference.llavaov_hermes import (
    LlavaOneVision_Hermes,
    load_model,
)

print("HERMES import successful")

HERMES import successful


download the 0.5B model

In [9]:
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path(
    "/kaggle/working/HERMES/models/"
    "llava-onevision-qwen2-0.5b-ov-hf"
)

model_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="llava-hf/llava-onevision-qwen2-0.5b-ov-hf",
    local_dir=str(model_dir),
)

print(model_dir)

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/428 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/kaggle/working/HERMES/models/llava-onevision-qwen2-0.5b-ov-hf


In [11]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from transformers import (
    LlavaOnevisionForConditionalGeneration,
    AutoProcessor,
    DynamicCache,
)

import decord
import logzero

sys.path.insert(0, "/kaggle/working/HERMES")
from inference.llavaov_hermes import load_model

print("\n=== EVERYTHING IMPORTED SUCCESSFULLY ===")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
Transformers: 4.45.0.dev0
CUDA: True
GPU: Tesla T4

=== EVERYTHING IMPORTED SUCCESSFULLY ===


In [12]:
%cd /kaggle/working/HERMES

from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path(
    "/kaggle/working/HERMES/models/"
    "llava-onevision-qwen2-0.5b-ov-hf"
)

model_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="llava-hf/llava-onevision-qwen2-0.5b-ov-hf",
    local_dir=str(model_dir),
)

print("Downloaded to:", model_dir)

/kaggle/working/HERMES


Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Downloaded to: /kaggle/working/HERMES/models/llava-onevision-qwen2-0.5b-ov-hf


In [13]:
!du -sh models/llava-onevision-qwen2-0.5b-ov-hf
!ls -lah models/llava-onevision-qwen2-0.5b-ov-hf | head -20

14G	models/llava-onevision-qwen2-0.5b-ov-hf
total 1.7G
drwxr-xr-x 5 root root 4.0K Sep 15 14:49 .
drwxr-xr-x 3 root root 4.0K Sep 15 14:49 ..
-rw-r--r-- 1 root root  122 Sep 15 14:47 added_tokens.json
drwxr-xr-x 3 root root 4.0K Sep 15 14:47 .cache
-rw-r--r-- 1 root root  826 Sep 15 14:47 chat_template.json
-rw-r--r-- 1 root root 2.6K Sep 15 14:47 config.json
-rw-r--r-- 1 root root  126 Sep 15 14:47 generation_config.json
-rw-r--r-- 1 root root 1.5K Sep 15 14:47 .gitattributes
-rw-r--r-- 1 root root 204K Sep 15 14:47 llava_onevision_arch.png
-rw-r--r-- 1 root root 1.6M Sep 15 14:47 merges.txt
-rw-r--r-- 1 root root 1.7G Sep 15 14:49 model.safetensors
drwxr-xr-x 2 root root 4.0K Sep 15 14:49 onnx
-rw-r--r-- 1 root root 1.7K Sep 15 14:49 preprocessor_config.json
-rw-r--r-- 1 root root  178 Sep 15 14:49 processor_config.json
-rw-r--r-- 1 root root 9.4K Sep 15 14:47 README.md
-rw-r--r-- 1 root root  367 Sep 15 14:49 special_tokens_map.json
-rw-r--r-- 1 root root 1.8K Sep 15 14:49 tokenizer

test loading the base LLaVA model

In [14]:
import torch
from transformers import (
    LlavaOnevisionForConditionalGeneration,
    AutoProcessor,
)

MODEL_PATH = (
    "/kaggle/working/HERMES/models/"
    "llava-onevision-qwen2-0.5b-ov-hf"
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully")
print("Main device:", model.device)

Model loaded successfully
Main device: cuda:0


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['image_newline']
  warnings.warn(


free the model before the HERMES test

In [15]:
del model
del processor

import gc
gc.collect()

torch.cuda.empty_cache()

!nvidia-smi

Tue Sep 15 14:52:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             27W /   70W |     105MiB /  15360MiB |     24%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

inspect HERMES's loader

In [16]:
import sys
import inspect

sys.path.insert(0, "/kaggle/working/HERMES")

from inference.llavaov_hermes import load_model

print(inspect.signature(load_model))
print(inspect.getsource(load_model))

(model_path='llava-onevision-qwen2-7b-ov-hf', n_init=None, kv_size=None, streaming=True, device='cuda', sample_fps=0.5)
def load_model(model_path='llava-onevision-qwen2-7b-ov-hf',
               n_init=None, kv_size=None, streaming=True, device="cuda", sample_fps=0.5):
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    processor.tokenizer.padding_side = 'left'
    
    system_prompt = '<|im_start|>system \nYou are a helpful assistant.<|im_end|><|im_start|>user '
    init_prompt_ids = processor.tokenizer(system_prompt, return_tensors="pt").input_ids.to(device)
    
    base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    model = LlavaOneVision_Hermes.__new__(LlavaOneVision_Hermes)
    model.__dict__ = base_model.__dict__.copy()

    Abstract_Hermes.__init__(
        model, 
        processor, 
        init_prompt_

In [17]:
!git rev-parse HEAD

8d699b16a6bedb9086c1b39ec4253c6a1d1ce789


In [18]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   inference/__pycache__/abstract_hermes.cpython-312.pyc
	modified:   inference/__pycache__/llavaov_hermes.cpython-312.pyc

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	inference/__pycache__/reindex_1d.cpython-312.pyc
	models/
	requirements_kaggle.txt

no changes added to commit (use "git add" and/or "git commit -a")


In [19]:
!find /kaggle/input -type f \
    \( -iname "*.mp4" -o -iname "*.mov" -o -iname "*.avi" \) \
    | head -20

/kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4


In [20]:
VIDEO = "/kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4"

In [21]:
from decord import VideoReader

vr = VideoReader(VIDEO)

num_frames = len(vr)
fps = float(vr.get_avg_fps())
duration = num_frames / fps

print("Frames:", num_frames)
print("FPS:", fps)
print("Duration:", round(duration, 2), "seconds")

Frames: 3392
FPS: 29.97002997002997
Duration: 113.18 seconds


inspect the expected HERMES annotation

In [22]:
import json

anno_path = "data/streamingbench/streamingbench_realtime.json"

with open(anno_path, "r") as f:
    official = json.load(f)

print(type(official))
print("Number of samples:", len(official))
print(json.dumps(official[0], indent=2)[:5000])

<class 'list'>
Number of samples: 498
{
  "video_id": 1,
  "video_path": "/data/streamingbench/videos/sample_416_real.mp4",
  "conversations": [
    {
      "question": "What can be seen in the cockpit's center screen right now?",
      "answer": "A map.",
      "choices": [
        "A compass.",
        "A map.",
        "An altitude meter.",
        "A weather radar."
      ],
      "end_time": 6.0,
      "task": "Object Recognition",
      "time_stamp": "00:00:06",
      "required_ability": "working memory"
    },
    {
      "question": "What colors are the pilot's handheld controls right now?",
      "answer": "Black.",
      "choices": [
        "Black.",
        "Red and green.",
        "Yellow and blue.",
        "Black and orange."
      ],
      "end_time": 157.0,
      "task": "Attribute Recognition",
      "time_stamp": "00:02:37",
      "required_ability": "working memory"
    },
    {
      "question": "What are the visible weather conditions right now?",
      "answer":

In [24]:
%cd /kaggle/working/HERMES

/kaggle/working/HERMES


In [25]:
%cd /kaggle/working/HERMES

/kaggle/working/HERMES


In [27]:
%cd /kaggle/working/HERMES

!ls -l data/kaggle_smoke.json

/kaggle/working/HERMES
ls: cannot access 'data/kaggle_smoke.json': No such file or directory


In [29]:
from pathlib import Path

videos = list(Path("/kaggle/input").rglob("*.mp4"))

print("Found videos:")
for i, v in enumerate(videos):
    print(i, v)

Found videos:
0 /kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4


In [30]:
VIDEO = str(videos[0])
print(VIDEO)

/kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4


In [31]:
from decord import VideoReader

vr = VideoReader(VIDEO)

fps = float(vr.get_avg_fps())
num_frames = len(vr)
duration = num_frames / fps

print("FPS:", fps)
print("Frames:", num_frames)
print("Duration:", round(duration, 2), "seconds")

FPS: 29.97002997002997
Frames: 3392
Duration: 113.18 seconds


In [32]:
import json
from pathlib import Path

Path("data").mkdir(exist_ok=True)

# Choose three positions in the video
t1 = duration * 0.25
t2 = duration * 0.55
t3 = duration * 0.90

def timestamp(seconds):
    seconds = int(seconds)
    minutes = seconds // 60
    secs = seconds % 60
    return f"00:{minutes:02d}:{secs:02d}"

annotation = [
    {
        "video_id": "kaggle_test_001",
        "video_path": VIDEO,
        "conversations": [
            {
                "question": "What is happening in the video right now?",
                "answer": "",
                "end_time": t1,
                "task": "Action Recognition",
                "time_stamp": timestamp(t1),
                "required_ability": "working memory"
            },
            {
                "question": "What is the person doing at this point in the video?",
                "answer": "",
                "end_time": t2,
                "task": "Action Recognition",
                "time_stamp": timestamp(t2),
                "required_ability": "working memory"
            },
            {
                "question": "Summarize the important events that have happened so far.",
                "answer": "",
                "end_time": t3,
                "task": "Event Understanding",
                "time_stamp": timestamp(t3),
                "required_ability": "long-term memory"
            }
        ]
    }
]

with open("data/kaggle_smoke.json", "w") as f:
    json.dump(annotation, f, indent=2)

print("Created data/kaggle_smoke.json")
print(json.dumps(annotation, indent=2))

Created data/kaggle_smoke.json
[
  {
    "video_id": "kaggle_test_001",
    "video_path": "/kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4",
    "conversations": [
      {
        "question": "What is happening in the video right now?",
        "answer": "",
        "end_time": 28.294933333333333,
        "task": "Action Recognition",
        "time_stamp": "00:00:28",
        "required_ability": "working memory"
      },
      {
        "question": "What is the person doing at this point in the video?",
        "answer": "",
        "end_time": 62.24885333333334,
        "task": "Action Recognition",
        "time_stamp": "00:01:02",
        "required_ability": "working memory"
      },
      {
        "question": "Summarize the important events that have happened so far.",
        "answer": "",
        "end_time": 101.86176,
        "task": "Event Understanding",
        "time_stamp": "00:01:41",
        "required_ability": "long-term memory"
      }
    ]
  }
]


In [33]:
!ls -lh data/kaggle_smoke.json

-rw-r--r-- 1 root root 968 Sep 15 15:06 data/kaggle_smoke.json


In [34]:
with open("data/kaggle_smoke.json") as f:
    test = json.load(f)

print("Videos:", len(test))
print("Video path:", test[0]["video_path"])
print("Questions:", len(test[0]["conversations"]))

for q in test[0]["conversations"]:
    print(q["end_time"], "->", q["question"])

Videos: 1
Video path: /kaggle/input/datasets/mohammadhameem/videoplayback/videoplayback.mp4
Questions: 3
28.294933333333333 -> What is happening in the video right now?
62.24885333333334 -> What is the person doing at this point in the video?
101.86176 -> Summarize the important events that have happened so far.


In [37]:
!CUDA_VISIBLE_DEVICES=0 \
python -m video_qa.hermes_vqa \
    --model llava_ov_0.5b \
    --sample_fps 0.5 \
    --save_dir results/kaggle_smoke \
    --anno_path data/kaggle_smoke.json \
    --debug true \
    --num_chunks 1 \
    --chunk_idx 0 \
    --kv_size 500 \
    --streaming true

[I 260915 15:08:56 base:273] seed: 2024
[I 260915 15:08:56 base:278] Loading VideoQA model: models/llava-onevision-qwen2-0.5b-ov-hf
[I 260915 15:08:59 llavaov_hermes:930] n_init: 13
[I 260915 15:08:59 llavaov_hermes:931] kv_size: 500
  0%|                                                     | 0/1 [00:00<?, ?it/s][D 260915 15:08:59 base:232] video_id: kaggle_test_001

  0%|                                                     | 0/3 [00:00<?, ?it/s][D 260915 15:09:01 hermes_vqa:44] sample: {'question': 'What is happening in the video right now?', 'answer': '', 'end_time': 28.294933333333333, 'task': 'Action Recognition', 'time_stamp': '00:00:28', 'required_ability': 'working memory'}
Encoding frames 0 to 14
[I 260915 15:09:02 hermes_vqa:61] Triggering question prediction and KV compression
Local question: Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships.
Global question: Summarize the video narrative, identifying main char

run the paper-style 4K configuration

In [38]:
%cd /kaggle/working/HERMES

!rm -rf results/kaggle_4k

!CUDA_VISIBLE_DEVICES=0 \
python -m video_qa.hermes_vqa \
    --model llava_ov_0.5b \
    --sample_fps 0.5 \
    --save_dir results/kaggle_4k \
    --anno_path data/kaggle_smoke.json \
    --debug true \
    --num_chunks 1 \
    --chunk_idx 0 \
    --kv_size 4000 \
    --streaming true

/kaggle/working/HERMES
[I 260915 15:11:05 base:273] seed: 2024
[I 260915 15:11:05 base:278] Loading VideoQA model: models/llava-onevision-qwen2-0.5b-ov-hf
[I 260915 15:11:08 llavaov_hermes:930] n_init: 13
[I 260915 15:11:08 llavaov_hermes:931] kv_size: 4000
  0%|                                                     | 0/1 [00:00<?, ?it/s][D 260915 15:11:08 base:232] video_id: kaggle_test_001

  0%|                                                     | 0/3 [00:00<?, ?it/s][D 260915 15:11:10 hermes_vqa:44] sample: {'question': 'What is happening in the video right now?', 'answer': '', 'end_time': 28.294933333333333, 'task': 'Action Recognition', 'time_stamp': '00:00:28', 'required_ability': 'working memory'}
Encoding frames 0 to 14
[I 260915 15:11:11 hermes_vqa:61] Triggering question prediction and KV compression
Local question: Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships.
Global question: Summarize the video narrativ

uncompressed control

In [39]:
%cd /kaggle/working/HERMES

!rm -rf results/kaggle_no_compression

!CUDA_VISIBLE_DEVICES=0 \
python -m video_qa.hermes_vqa \
    --model llava_ov_0.5b \
    --sample_fps 0.5 \
    --save_dir results/kaggle_no_compression \
    --anno_path data/kaggle_smoke.json \
    --debug true \
    --num_chunks 1 \
    --chunk_idx 0 \
    --kv_size 50000 \
    --streaming true

/kaggle/working/HERMES
[I 260915 15:12:02 base:273] seed: 2024
[I 260915 15:12:02 base:278] Loading VideoQA model: models/llava-onevision-qwen2-0.5b-ov-hf
[I 260915 15:12:05 llavaov_hermes:930] n_init: 13
[I 260915 15:12:05 llavaov_hermes:931] kv_size: 50000
  0%|                                                     | 0/1 [00:00<?, ?it/s][D 260915 15:12:05 base:232] video_id: kaggle_test_001

  0%|                                                     | 0/3 [00:00<?, ?it/s][D 260915 15:12:07 hermes_vqa:44] sample: {'question': 'What is happening in the video right now?', 'answer': '', 'end_time': 28.294933333333333, 'task': 'Action Recognition', 'time_stamp': '00:00:28', 'required_ability': 'working memory'}
Encoding frames 0 to 14
[I 260915 15:12:08 hermes_vqa:61] Triggering question prediction and KV compression
Local question: Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships.
Global question: Summarize the video narrati

the results

In [40]:
import pandas as pd

r500 = pd.read_csv("results/kaggle_smoke/1_0.csv")
r4000 = pd.read_csv("results/kaggle_4k/1_0.csv")
r50000 = pd.read_csv("results/kaggle_no_compression/1_0.csv")

print("=== KV 500 ===")
display(r500)

print("=== KV 4000 ===")
display(r4000)

print("=== KV 50000 ===")
display(r50000)

=== KV 500 ===


,video_id,question,answer,pred_answer,task
0,kaggle_test_001,What is happening in the video right now?,NaN,"The video is set in a kitchen, where the sandw...",Action Recognition
1,kaggle_test_001,What is the person doing at this point in the ...,NaN,"The person is preparing a sandwich, specifical...",Action Recognition
2,kaggle_test_001,Summarize the important events that have happe...,NaN,The video begins with a close-up of a sandwich...,Event Understanding


=== KV 4000 ===


,video_id,question,answer,pred_answer,task
0,kaggle_test_001,What is happening in the video right now?,NaN,The video is showing a close-up of a sandwich ...,Action Recognition
1,kaggle_test_001,What is the person doing at this point in the ...,NaN,The person is preparing a sandwich by adding i...,Action Recognition
2,kaggle_test_001,Summarize the important events that have happe...,NaN,The video begins with a close-up of a sandwich...,Event Understanding


=== KV 50000 ===


,video_id,question,answer,pred_answer,task
0,kaggle_test_001,What is happening in the video right now?,NaN,The video is showing a close-up of a sandwich ...,Action Recognition
1,kaggle_test_001,What is the person doing at this point in the ...,NaN,The person is preparing a sandwich by adding i...,Action Recognition
2,kaggle_test_001,Summarize the important events that have happe...,NaN,The video begins with a close-up of a sandwich...,Event Understanding


In [41]:
import pandas as pd

runs = {
    "KV500": "results/kaggle_smoke/1_0.csv",
    "KV4000": "results/kaggle_4k/1_0.csv",
    "KV50000": "results/kaggle_no_compression/1_0.csv",
}

dfs = {name: pd.read_csv(path) for name, path in runs.items()}

for i in range(len(dfs["KV500"])):
    print("=" * 100)
    print("QUESTION:")
    print(dfs["KV500"].iloc[i]["question"])

    for name, df in dfs.items():
        print(f"\n{name}:")
        print(df.iloc[i]["pred_answer"])

QUESTION:
What is happening in the video right now?

KV500:
The video is set in a kitchen, where the sandwich is being prepared.

KV4000:
The video is showing a close-up of a sandwich with various fillings, including grilled chicken and vegetables.

KV50000:
The video is showing a close-up of a sandwich with various fillings, including grilled chicken and vegetables.
QUESTION:
What is the person doing at this point in the video?

KV500:
The person is preparing a sandwich, specifically making the filling for it.

KV4000:
The person is preparing a sandwich by adding ingredients to the sandwich and setting it aside.

KV50000:
The person is preparing a sandwich by adding ingredients to the sandwich, including sautéed shallots and black pepper.
QUESTION:
Summarize the important events that have happened so far.

KV500:
The video begins with a close-up of a sandwich being prepared, followed by a shot of the sandwich on a wooden table. The scene transitions to a close-up of a yellow food item